In [ ]:
# Core
import pandas as pd
import numpy as np

# Models
from sklearn.tree import DecisionTreeRegressor

# Metrics
from sklearn.metrics import mean_squared_error

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("data/electricity_market_dataset.csv")
# Convert timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort by time (VERY IMPORTANT)
df = df.sort_values('Timestamp').reset_index(drop=True)

df.head()
print("Shape:", df.shape)


Format the data

In [ ]:
target = "Electricity_Price_Forecast"
df_model = df.copy()

# Encode categorical columns
df_model["Regulatory_Policies"] = df_model["Regulatory_Policies"].map({"Low": 0, "Medium": 1, "High": 2})
df_model["Energy_Access_Data"] = df_model["Energy_Access_Data"].map({"Rural": 0, "Urban": 1})
df_model["Project_Risk_Analysis"] = df_model["Project_Risk_Analysis"].map({"Low": 0, "Medium": 1, "High": 2})


# Drop target
X = df_model.drop(columns=[target, "Timestamp"])
y = df_model[target]

RMSE Function for later user

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
def get_time_splits(data, look_back, look_ahead, step=24):
    """
    look_back and look_ahead are in number of rows (hours)
    """
    splits = []
    
    for i in range(look_back, len(data) - look_ahead, step):
        train_start = i - look_back
        train_end = i
        
        test_start = i
        test_end = i + look_ahead
        
        splits.append((train_start, train_end, test_start, test_end))
    
    return splits

Define Static Function

In [ ]:
# might not need this
def run_static_model(X, y, look_back, look_ahead):
    errors = []
    
    splits = get_time_splits(X, look_back, look_ahead)
    
    for (train_start, train_end, test_start, test_end) in splits:
        
        X_train = X.iloc[train_start:train_end]
        y_train = y.iloc[train_start:train_end]
        
        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]
        
        model = DecisionTreeRegressor(max_depth=5)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        error = rmse(y_test, preds)
        
        errors.append(error)
    
    return np.mean(errors)
def train_static(X, y, look_back, depthOfTree):
    X_train = X.iloc[:look_back]
    y_train = y.iloc[:look_back]

    X_test = X.iloc[look_back:]
    y_test = y.iloc[look_back:]

    model = DecisionTreeRegressor(max_depth=depthOfTree)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    return predictions, y_test


Run fist experiment

In [ ]:
# Training window sizes (1, 3, 6, 12, 18, 24 months)
look_back_values = [720, 2160, 4320, 8640,  12960, 17280]  
tree_depths = [3, 5, 7, 10]  

# Dictionary to store RMSE
results = {}

for lb in look_back_values:
    results[lb] = {}  # inner dict for tree depths
    for depth in tree_depths:
        # Train static model and predict remaining data
        predictions, y_test = train_static(X, y, look_back=lb, depthOfTree=depth)
        
        # Compute RMSE
        error = np.sqrt(mean_squared_error(y_test, predictions))
        
        # Store
        results[lb][depth] = error
        
        print(f"Look-back {lb}, Depth {depth} → RMSE: {error:.2f}")
df_results = pd.DataFrame(results).T  # look-back as rows, depths as columns
df_results

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_results.astype(float), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Static Model RMSE: Look-Back vs Tree Depth")
plt.xlabel("Tree Depth")
plt.ylabel("Look-Back (rows)")
plt.show()

Defining Progressive Training Function